#Data Acquisition Casestudy and Intermediate Assessment 2

##Step 1: Load SpaceX Launch Data from API


In [10]:
import requests
import pandas as pd

load_url = "https://api.spacexdata.com/v4/launches"
response = requests.get(load_url)

# Convert JSON to Python list
data = response.json()

load_df = pd.DataFrame(data)

###● Extract relevant columns: name, date_utc, success, details, rocket
###● Convert date_utc to datetime and extract the year

In [11]:
load_df = df[["name", "date_utc", "success", "details", "rocket"]]
load_df["date_utc"] = pd.to_datetime(load_df["date_utc"]) #Convert date_utc to datetime
load_df["year"] = load_df["date_utc"].dt.year #Extract year
load_df.head(10)

/tmp/ipykernel_6628/2640091455.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  load_df["date_utc"] = pd.to_datetime(load_df["date_utc"]) #Convert date_utc to datetime


,name,date_utc,success,details,rocket,year
0,FalconSat,2006-03-24 22:30:00+00:00,False,Engine failure at 33 seconds and loss of vehicle,5e9d0d95eda69955f709d1eb,2006
1,DemoSat,2007-03-21 01:10:00+00:00,False,Successful first stage burn and transition to ...,5e9d0d95eda69955f709d1eb,2007
2,Trailblazer,2008-08-03 03:34:00+00:00,False,Residual stage 1 thrust led to collision betwe...,5e9d0d95eda69955f709d1eb,2008
3,RatSat,2008-09-28 23:15:00+00:00,True,Ratsat was carried to orbit on the first succe...,5e9d0d95eda69955f709d1eb,2008
4,RazakSat,2009-07-13 03:35:00+00:00,True,None,5e9d0d95eda69955f709d1eb,2009
5,Falcon 9 Test Flight,2010-06-04 18:45:00+00:00,True,None,5e9d0d95eda69973a809d1ec,2010
6,COTS 1,2010-12-08 15:43:00+00:00,True,None,5e9d0d95eda69973a809d1ec,2010
7,COTS 2,2012-05-22 07:44:00+00:00,True,"Launch was scrubbed on first attempt, second l...",5e9d0d95eda69973a809d1ec,2012
8,CRS-1,2012-10-08 00:35:00+00:00,True,"CRS-1 successful, but the secondary payload wa...",5e9d0d95eda69973a809d1ec,2012
9,CRS-2,2013-03-01 19:10:00+00:00,True,Last launch of the original Falcon 9 v1.0 laun...,5e9d0d95eda69973a809d1ec,2013


##Step 2: Load Rocket Metadata

In [12]:
rocket_url = "https://api.spacexdata.com/v4/rockets"
response = requests.get(rocket_url)

# Convert JSON to Python list
rocket_data = response.json()

#2: Convert to DataFrame
rocket_df = pd.DataFrame(rocket_data)

###Extract id, name, type, active, and stages


In [19]:
rocket_df = rocket_df[["id", "name", "type", "active", "stages"]]
print(rocket_df.head())
print(rocket_df.info())
print(rocket_df)

                         id          name    type  active  stages
0  5e9d0d95eda69955f709d1eb      Falcon 1  rocket   False       2
1  5e9d0d95eda69973a809d1ec      Falcon 9  rocket    True       2
2  5e9d0d95eda69974db09d1ed  Falcon Heavy  rocket    True       2
3  5e9d0d96eda699382d09d1ee      Starship  rocket   False       2
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      4 non-null      object
 1   name    4 non-null      object
 2   type    4 non-null      object
 3   active  4 non-null      bool  
 4   stages  4 non-null      int64 
dtypes: bool(1), int64(1), object(3)
memory usage: 264.0+ bytes
None
                         id          name    type  active  stages
0  5e9d0d95eda69955f709d1eb      Falcon 1  rocket   False       2
1  5e9d0d95eda69973a809d1ec      Falcon 9  rocket    True       2
2  5e9d0d95eda69974db09d1ed  Falcon Heavy  rocket  

##Step 3: Merge Launch and Rocket Data


###Join the two DataFrames on rocket ID using


In [21]:
merge_df = pd.merge(df, rocket_df, left_on='rocket', right_on='id', how="left")

print("Merged Launch and Rocket Data:\n") # Display merged data
print(merge_df.head(15))

Merged Launch and Rocket Data:

                  name_x                  date_utc success  \
0              FalconSat 2006-03-24 22:30:00+00:00   False   
1                DemoSat 2007-03-21 01:10:00+00:00   False   
2            Trailblazer 2008-08-03 03:34:00+00:00   False   
3                 RatSat 2008-09-28 23:15:00+00:00    True   
4               RazakSat 2009-07-13 03:35:00+00:00    True   
5   Falcon 9 Test Flight 2010-06-04 18:45:00+00:00    True   
6                 COTS 1 2010-12-08 15:43:00+00:00    True   
7                 COTS 2 2012-05-22 07:44:00+00:00    True   
8                  CRS-1 2012-10-08 00:35:00+00:00    True   
9                  CRS-2 2013-03-01 19:10:00+00:00    True   
10              CASSIOPE 2013-09-29 16:00:00+00:00    True   
11                 SES-8 2013-12-03 22:41:00+00:00    True   
12             Thaicom 6 2014-01-06 18:06:00+00:00    True   
13                 CRS-3 2014-04-18 19:25:00+00:00    True   
14        OG-2 Mission 1 2014-07-14 15

##Step 4: Add Simulated Country Information

###Add a new column country and randomly assign one of these values:
####['USA', 'Russia', 'India', 'China', 'France']


In [28]:
import random

countries = ['USA', 'Russia', 'India', 'China', 'France']
merge_df['country'] = [random.choice(countries) for _ in range(len(merge_df))]
merge_df.head()

,name_x,date_utc,success,details,rocket,year,id,name_y,type,active,stages,country
0,FalconSat,2006-03-24 22:30:00+00:00,False,Engine failure at 33 seconds and loss of vehicle,5e9d0d95eda69955f709d1eb,2006,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2,India
1,DemoSat,2007-03-21 01:10:00+00:00,False,Successful first stage burn and transition to ...,5e9d0d95eda69955f709d1eb,2007,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2,France
2,Trailblazer,2008-08-03 03:34:00+00:00,False,Residual stage 1 thrust led to collision betwe...,5e9d0d95eda69955f709d1eb,2008,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2,India
3,RatSat,2008-09-28 23:15:00+00:00,True,Ratsat was carried to orbit on the first succe...,5e9d0d95eda69955f709d1eb,2008,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2,France
4,RazakSat,2009-07-13 03:35:00+00:00,True,None,5e9d0d95eda69955f709d1eb,2009,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2,India


##Step 5: Store Merged Data in SQLite3


###● Use sqlite3 to create a connection and save the merged DataFrame as a table named launches
###● Table should contain all merged columns including country

In [60]:
import sqlite3
conn = sqlite3.connect("spacex.db")

merge_df.to_sql("launches", conn, if_exists="replace", index=False)
print(f"Table 'launches' created with {total_records} records")

Table 'launches' created with 205 records


In [61]:
#Save the merged DataFrame to a table named 'launches'

cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM launches")
total_records = cursor.fetchone()[0]

print("Data successfully saved to 'spacex.db' as table 'launches'.")

Data successfully saved to 'spacex.db' as table 'launches'.


In [63]:
cursor.execute("SELECT * FROM launches LIMIT 1")
column_names = [desc[0] for desc in cursor.description]
print(f"Columns: {', '.join(column_names)}")

Columns: name_x, date_utc, success, details, rocket, year, id, name_y, type, active, stages, country


##Step 6: Run SQL Queries on the Data to analyze


###1. Launches by Country
###2. Which year had the highest number of launches?
###3. Top 5 Missions by Launch Count

In [47]:
conn = sqlite3.connect('spacex.db')

In [48]:
q1 = """
SELECT country, COUNT(*) AS total_launches
FROM launches
GROUP BY country
ORDER BY total_launches DESC;
"""

print(pd.read_sql(q1, conn))

  country  total_launches
0  France              47
1  Russia              42
2     USA              40
3   India              39
4   China              37


In [49]:
q2 = """
SELECT year, COUNT(*) AS Highest_launches
FROM launches
GROUP BY year
ORDER BY Highest_launches DESC
LIMIT 1;
"""

print(pd.read_sql(q2, conn))

   year  Highest_launches
0  2022                62


In [50]:
q3 = """
SELECT name_x, COUNT(*) AS launch_count
FROM launches
GROUP BY name_x
ORDER BY launch_count DESC
LIMIT 5;
"""

print(pd.read_sql(q3, conn))

                      name_x  launch_count
0  ispace Mission 1 & Rashid             1
1                       ZUMA             1
2     WorldView Legion 1 & 2             1
3        Viasat-3 & Arcturus             1
4                    USSF-44             1


In [65]:
conn.close()